### Carregar Bases

In [128]:
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split

_img_size = (48,48)

def carregar_base_de_dados(caminho_principal, lista_classes, proporcao_teste):
    """
    Varre as subpastas no caminho_principal, processa as imagens e 
    retorna matrizes prontas para o treinamento da rede neural.
    """
    imagens_processadas = []
    labels_numéricos = []
    
    # Criamos um dicionário para mapear o nome da pasta para um número (ex: "Semibreve" -> 0)
    mapeamento_classes = {nome_classe: idx for idx, nome_classe in enumerate(lista_classes)}
    
    print("🔄 Iniciando o carregamento e processamento das imagens...")
    
    # 1. Percorre cada pasta de classe listada
    for nome_classe in lista_classes:
        caminho_pasta = os.path.join(caminho_principal, nome_classe)
        
        # Verifica se a pasta realmente existe para evitar erros
        if not os.path.isdir(caminho_pasta):
            print(f"⚠️ Aviso: A pasta '{nome_classe}' não foi encontrada em {caminho_principal}. Pulando...")
            continue
            
        codigo_classe = mapeamento_classes[nome_classe]
        
        # 2. Lista todos os arquivos dentro daquela subpasta
        for nome_arquivo in os.listdir(caminho_pasta):
            caminho_imagem = os.path.join(caminho_pasta, nome_arquivo)
            
            # Lê a imagem
            img = cv2.imread(caminho_imagem)
            if img is None:
                continue # Pula arquivos corrompidos ou que não sejam imagens
                
            # --- SEU PIPELINE DE TRATAMENTO DE IMAGEM ---
            # Converte para escala de cinza
            cinza = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            
            # Se a imagem não for 16x16, força o redimensionamento por segurança
            if cinza.shape != _img_size:
                cinza = cv2.resize(cinza, _img_size)
                
            # Pixels acima de 127 viram 255 (branco), abaixo viram 0 (preto)
            _, binaria = cv2.threshold(cinza, 127, 255, cv2.THRESH_BINARY)
            
            # Normaliza os valores para a escala entre 0.0 e 1.0
            X_normalizado = binaria / 255.0
            # ---------------------------------------------
            
            # Salva os dados processados nas listas
            imagens_processadas.append(X_normalizado)
            labels_numéricos.append(codigo_classe)
            
    # Converte as listas Python em arrays do NumPy eficientes
    X_dados = np.array(imagens_processadas)
    Y_dados = np.array(labels_numéricos)
    
    print(f"✅ Total carregado: {len(X_dados)} imagens.")
    
    # --- NOVA ETAPA: DIVISÃO EM TREINO E TESTE ---
    # random_state=42 garante que a divisão seja sempre a mesma ao rodar o script
    # stratify=Y_dados garante proporções iguais de classes no treino e teste
    X_train, X_test, Y_train, Y_test = train_test_split(
        X_dados, Y_dados, 
        test_size=proporcao_teste, 
        random_state=1, 
        stratify=Y_dados
    )
    
    print(f"📊 Divisão Concluída: Treino = {len(X_train)} | Teste = {len(X_test)}")
    return X_train, X_test, Y_train, Y_test

In [129]:
classes = ["Semibreve", "Minima", "Seminima", "Colcheia", "Semicolcheia"]
diretorio_base = r'C:\Users\jesse\SIN5016\dados_treino'

# Carrega e divide os dados
X_treino, X_teste, Y_treino, Y_teste = carregar_base_de_dados(diretorio_base, classes, proporcao_teste=0.2)

🔄 Iniciando o carregamento e processamento das imagens...
✅ Total carregado: 4250 imagens.
📊 Divisão Concluída: Treino = 3400 | Teste = 850


### Teste

In [130]:
def DistanciaEuclidiana_Teste(nova_imagem):
    distancias  = [] 
    classe_list = []
    nova_imagem_flat = nova_imagem.flatten()

    for index, imagem in enumerate(X_teste):
        dist = sum((nova_imagem_flat - imagem.flatten())**2.0)
        distancias.append(dist)
        classe_list.append(Y_teste[index])

    dist_classe = dict(zip(distancias, classe_list))
    dist_classe_sorted = sorted(dist_classe.items(), key=lambda item: item[0])

    return dist_classe_sorted

def K_NearestNeighbors_Teste(novo_email, alfa):
    Distancias_Classe = DistanciaEuclidiana_Teste(novo_email)

    # top (alfa) animais
    top_k_distancias = Distancias_Classe[:alfa]

    top_k_classes = []
    for i in top_k_distancias:
        top_k_classes.append(i[1])

    import statistics
    y = statistics.mode(top_k_classes)

    return y 

# Validando e Calculando a acurácia do Modelo
alfa = 5 
pontos = []

for i, teste in enumerate(X_teste):
    y_hat = K_NearestNeighbors_Teste(teste, alfa)

    if (y_hat == Y_teste[i]):
        pontos.append(1)
    else:
        pontos.append(0)

    acuracia_atual = np.mean(pontos) * 100
    if (i + 1) % 50 == 0 or i == 0:
        print(f'Processando Imagem {i + 1}/{len(X_teste)} | Acurácia atual: {acuracia_atual:.2f}%')

acuracia = np.mean(pontos) * 100
print(f"Acurácia: {acuracia:.2f}%")

Processando Imagem 1/850 | Acurácia atual: 100.00%
Processando Imagem 50/850 | Acurácia atual: 88.00%
Processando Imagem 100/850 | Acurácia atual: 85.00%
Processando Imagem 150/850 | Acurácia atual: 86.00%
Processando Imagem 200/850 | Acurácia atual: 86.00%
Processando Imagem 250/850 | Acurácia atual: 86.80%
Processando Imagem 300/850 | Acurácia atual: 87.67%
Processando Imagem 350/850 | Acurácia atual: 88.29%
Processando Imagem 400/850 | Acurácia atual: 88.25%
Processando Imagem 450/850 | Acurácia atual: 88.00%
Processando Imagem 500/850 | Acurácia atual: 87.80%
Processando Imagem 550/850 | Acurácia atual: 87.64%
Processando Imagem 600/850 | Acurácia atual: 87.33%
Processando Imagem 650/850 | Acurácia atual: 88.00%
Processando Imagem 700/850 | Acurácia atual: 88.29%
Processando Imagem 750/850 | Acurácia atual: 88.53%
Processando Imagem 800/850 | Acurácia atual: 88.50%
Processando Imagem 850/850 | Acurácia atual: 88.47%
Acurácia: 88.47%


### Classificar nova Imagem

In [131]:
def DistanciaEuclidiana(nova_imagem):
    distancias  = [] 
    classe_list = []
    nova_imagem_flat = nova_imagem.flatten()

    for index, imagem in enumerate(X_treino):
        dist = sum((nova_imagem_flat - imagem.flatten())**2.0)
        distancias.append(dist)
        classe_list.append(Y_treino[index])

    dist_classe = dict(zip(distancias, classe_list))
    dist_classe_sorted = sorted(dist_classe.items(), key=lambda item: item[0])

    return dist_classe_sorted

def K_NearestNeighbors(novo_email, alfa):
    Distancias_Classe = DistanciaEuclidiana(novo_email)

    # top (alfa) animais
    top_k_distancias = Distancias_Classe[:alfa]

    top_k_classes = []
    for i in top_k_distancias:
        top_k_classes.append(i[1])

    import statistics
    y = statistics.mode(top_k_classes)

    # atribuir nome da classe
    classe_predita = classes[y]

    return classe_predita

In [132]:
# === PREDIÇÃO EM UMA NOVA IMAGEM ===
caminho_nova_img = r'C:\Users\jesse\SIN5016\bases\semicolcheia_.jpg'
alfa = 5 

img = cv2.imread(caminho_nova_img)
cinza = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
if cinza.shape != _img_size:
    cinza = cv2.resize(cinza, _img_size)

_, binaria = cv2.threshold(cinza, 127, 255, cv2.THRESH_BINARY)
X_novo = ((binaria / 255.0)).flatten() #.reshape(1, -1)

classe_predicao = K_NearestNeighbors(X_novo, alfa)

print(f"O e-mail foi classificado como: {classe_predicao}")

O e-mail foi classificado como: Semicolcheia
